In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, StackingClassifier

from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix, accuracy_score
from xgboost import XGBClassifier

df = pd.read_excel("ebola_contact.xlsx") 
X = df.drop(columns=['yn_outcome'])
y = df['yn_outcome']
print(df.shape)
numeric_features = X.columns.tolist()
preprocessor = ColumnTransformer(transformers=[('num', StandardScaler(), numeric_features)],remainder='passthrough')
rf = RandomForestClassifier(n_estimators=100, max_depth=12, min_samples_split=5, class_weight='balanced', random_state=42)
xgb = XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, 
                    scale_pos_weight=(y.value_counts()[0] / y.value_counts()[1]), eval_metric='logloss', random_state=42)
lr = Pipeline(steps=[('scaler', StandardScaler()),('lr', LogisticRegression(max_iter=1000, class_weight='balanced', solver='liblinear'))])
stack_model = StackingClassifier(estimators=[('rf', rf), ('xgb', xgb), ('lr', lr)],final_estimator=LogisticRegression(max_iter=1000, 
                class_weight='balanced', solver='liblinear'),stack_method='predict_proba',n_jobs=-1)
X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=0.2,stratify=y,random_state=42)
stack_model.fit(X_train, y_train)
y_pred = stack_model.predict(X_test)
y_prob = stack_model.predict_proba(X_test)[:, 1]
print("\nEnsemble Accuracy:", accuracy_score(y_test, y_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred, digits=4))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

joblib.dump(stack_model, 'stack_model.pkl')

print("Stacking model saved successfully!")



new_person = {
    'contacttype1': 1, 'contacttype2': 0, 'contacttype3': 0, 'contacttype4': 0, 
    'contacttype5': 0, 'contacttype6': 0, 'CountOfdayfollowup1': 5, 'contacttypes': 1, 
    'timediff': 2, 'phase': 2, 'SumOfpainjoint1': 2, 'SumOffever1': 3, 'SumOfweakness1': 2,
    'SumOfnausea1': 1, 'SumOfdiarrhea1': 1, 'SumOfheadache1': 2, 'SumOfpainthroat1': 0, 
    'SumOfredeye1': 0, 'SumOfbleedany1': 0, 'died': 0, 'missing_contacttype1': 0, 
    'missing_contacttype2': 0, 'missing_contacttype3': 0, 'missing_contacttype4': 0, 
    'missing_contacttype5': 0, 'missing_contacttype6': 0, 'total_symptom_types': 5, 
    'symptomatic': 1, 'yn_painjoint': 1, 'yn_fever': 1, 'yn_weakness': 1, 'yn_nausea': 1, 
    'yn_diarrhea': 1, 'yn_headache': 1, 'yn_painthroat': 0, 'yn_redeye': 0, 
    'yn_bleedany': 0, 'EW': 38, '4-9 symptoms': 1, 'total_symptom_days': 7
}
new_df = pd.DataFrame([new_person])
new_df = new_df[X.columns]
probability = stack_model.predict_proba(new_df)[0][1]
prediction = int(probability >= 0.3)
print("Ebola Positive Probability:", round(probability, 4))
print("Predicted Outcome:", "POSITIVE" if prediction == 1 else "NEGATIVE")
if probability >= 0.7:
    risk = "HIGH RISK - Immediate isolation"
elif probability >= 0.3:
    risk = "MEDIUM RISK - Daily monitoring"
else:
    risk = "LOW RISK - Routine follow-up"
print("Risk Level:", risk)



(6000, 41)

Ensemble Accuracy: 0.9891666666666666
Confusion Matrix:
[[591   9]
 [  4 596]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9933    0.9850    0.9891       600
           1     0.9851    0.9933    0.9892       600

    accuracy                         0.9892      1200
   macro avg     0.9892    0.9892    0.9892      1200
weighted avg     0.9892    0.9892    0.9892      1200

ROC-AUC: 0.9990777777777778
Stacking model saved successfully!
Ebola Positive Probability: 0.9865
Predicted Outcome: POSITIVE
Risk Level: HIGH RISK - Immediate isolation
